In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
import plotly.express as px
import sys
sys.path.append(str(Path.home() / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'/'config'))
import plot as pt



def tidy_vmt_3(xlsx_path):

    '''
    Function to reshape the wide format of the VMT_3 pdf report analyzer output
    '''

    # Read raw sheet without assuming a clean header ---------------------------------------------------------------------

    # The sheet has:
    # row 2 = COUNTY / JURISDICTION / years
    # row 3 = metric names: Maintained Miles / Daily VMT (1000s)
    
    sheet_name = "Jurisdiction (Table 6)"
    df = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None, engine="openpyxl")

    year_header_row = 2
    metric_header_row = 3
    data_start_row = 4

    # Build clean column names ---------------------------------------------------------------------

    years = df.iloc[year_header_row].ffill()
    metrics = df.iloc[metric_header_row]

    df_cols = pd.DataFrame(data={'years': years, 'metrics': metrics})
    df_cols['new_cols'] = df_cols['years'].astype(str) + '|' + df_cols['metrics'].fillna('0').astype(str)
    df_cols['new_cols'] = df_cols['new_cols'].str.replace('|0', '')
    new_cols = df_cols['new_cols'].tolist()


    # Keep data rows only ---------------------------------------------------------------------

    # Drop any fully empty columns that may have come through

    df_wide = df.iloc[data_start_row:].copy()
    df_wide.columns = new_cols
    df_wide = df_wide.loc[:, ~df_wide.columns.astype(str).str.startswith("drop_")]


    # Identify main jurisdiction rows vs. Other Agencies rows ---------------------------------------------------------------------

    # Mark "Other Agencies" row
    # Remove the marker row itself
    # Remove rows without a county/jurisdiction
    # Clean string columns

    other_agencies_mask = df_wide["COUNTY"].astype(str).str.strip().eq("Other Agencies")

    if not other_agencies_mask.any():
        raise ValueError("Could not find the 'Other Agencies' marker row.")

    other_agencies_marker_idx = df_wide.index[other_agencies_mask][0]
    df_wide["Agency Type"] = np.where(df_wide.index > other_agencies_marker_idx, "Other Agency", "Main Jurisdiction")

    df_wide = df_wide.loc[~other_agencies_mask].copy()
    df_wide = df_wide.dropna(subset=["COUNTY", "JURISDICTION"], how="all")
    df_wide["COUNTY"      ] = df_wide["COUNTY"      ].astype(str).str.strip()
    df_wide["JURISDICTION"] = df_wide["JURISDICTION"].astype(str).str.strip()


    # Melt from wide to long ---------------------------------------------------------------------

    # Split combined column name into Year + metric

    df_long = df_wide.melt(id_vars=["Agency Type", "COUNTY", "JURISDICTION"], var_name="Year_Metric", value_name="value")
    df_long[["Year", "Metric"]] = df_long["Year_Metric"].str.split("|", expand=True)
    df_long["Year"] = df_long["Year"].astype(int)


    # Pivot metrics back into separate columns ---------------------------------------------------------------------

    # Remove the columns index name created by pivot_table
    # Ensure desired columns exist and order them
    # Convert numeric columns
    # Drop year rows where both metrics are blank
    # Sort for readability

    df_tidy = (
        df_long
        .pivot_table(index=["Agency Type", "COUNTY", "JURISDICTION", "Year"], columns="Metric", values="value", aggfunc="first")
        .reset_index()
    )

    df_tidy.columns.name = None
    desired_cols = ["Agency Type", "COUNTY", "JURISDICTION", "Year", "Maintained Miles", "Daily VMT (1000s)"]
    df_tidy = df_tidy[desired_cols]

    df_tidy["Maintained Miles"] = pd.to_numeric(df_tidy["Maintained Miles"], errors="coerce")
    df_tidy["Daily VMT (1000s)"] = pd.to_numeric(df_tidy["Daily VMT (1000s)"], errors="coerce")

    df_tidy = df_tidy.dropna(subset=["Maintained Miles", "Daily VMT (1000s)"], how="all")
    df_tidy = df_tidy.sort_values(["Agency Type", "COUNTY", "JURISDICTION", "Year"], kind="stable").reset_index(drop=True)


    # Check ---------------------------------------------------------------------

    display(df_tidy.head())
    print(df_tidy.shape)
    print(df_tidy["Agency Type"].value_counts(dropna=False))

    return df_tidy



PATH_PLOTS = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring")
print('Export Location: ' + str(PATH_PLOTS))

pd.options.display.float_format = '{:.2f}'.format


EXPORT=False



indicator = 'VMT_3'
plot_name = 'daily'

xlsx_path = PATH_PLOTS/"Data"/"VMT_3 HPMS.xlsx"
df_tidy = tidy_vmt_3(xlsx_path)

# df_plot = df_tidy[df_tidy['Agency Type']=='Main Jurisdiction'].groupby(['COUNTY', 'Year'], as_index=False).agg(Daily_VMT_1ks=('Daily VMT (1000s)', 'sum'))
df_plot = df_tidy.groupby(['COUNTY', 'Year'], as_index=False).agg(Daily_VMT_1ks=('Daily VMT (1000s)', 'sum'))
df_plot['COUNTY'] = df_plot['COUNTY'].map({'EL DORADO':'El Dorado', 'PLACER':'Placer', 'SACRAMENTO':'Sacramento', 'SUTTER':'Sutter', 'YOLO':'Yolo', 'YUBA':'Yuba'})
df_plot = df_plot.sort_values(['Year', 'Daily_VMT_1ks'], ascending=[True, False]).reset_index(drop=True)

fig = px.bar(df_plot, x='Year', y='Daily_VMT_1ks', color='COUNTY', color_discrete_map=pt.colors_counties)
title = "<b>Total Daily VMT by County (Thousands)</b>"
fig.update_yaxes(tickformat=',.0f')
fig.update_xaxes(tick0=0, dtick=5)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})

pt.plot_agol(fig, EXPORT, title, indicator, plot_name)



In [ ]:


# indicator = 'VMT_3'
# plot_name = 'daily'

# df = pd.read_excel(PATH_PLOTS/'Data'/f"{indicator} HPMS.xlsx", sheet_name='Tidy_T6')
# df = df[df['JURISDICTION'].str.contains('TOTAL')]
# df['COUNTY'] = df['COUNTY'].map(pt.var_counties)
# df = df.sort_values(['DVMT_TOTAL', 'YEAR'], ascending=[False, True])
# display(df.head())

# fig = px.bar(df, x='YEAR', y='DVMT_TOTAL', color='COUNTY', color_discrete_map=pt.colors_counties)
# title = "<b>Total Daily VMT by County (Thousands)</b>"
# fig.update_yaxes(tickformat=',.0f')
# fig.update_xaxes(tick0=0, dtick=5)
# fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(legend={'traceorder': 'reversed'})

# pt.plot_agol(fig, EXPORT, title, indicator, plot_name)

